# Fase 2 · Transformación de datos en los datasets

## Objetivo

El objetivo de esta fase consiste en ejecutar los cambios y ajustes detectados durante el Análisis Exploratorio de Datos (EDA), para unificar, limpiar y transformar los datasets seleccionados en este proyecto.

En esta etapa se trabaja principalmente en:

- abordar los duplicados,
- combinar varios datasets,
- gestionar los valores nulos,
- homogenizar categorías para asegurar la integridad semántica,
- y exportar los archivos finales ya depurados.

Este proceso permitirá tener un conjunto de datos más consistentes, comparables y listos para la siguiente fase de análisis avanzado y visualización.

In [48]:
# Importación de librerías
import pandas as pd
import numpy as np
import re

# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación de módulos de transformación 
from src.etl.load_data import load_friends_data_raw, load_friends_data_translated
from src.etl import transform as trans
from transformers import pipeline
from src.etl import column_standardizer as stan
from src.etl import dataset_sanitizer as pr
from src.etl import soporte_correlacion as sop



 
# Configuración para visualizar todas las columnas del DataFrame
pd.set_option('display.max_columns', None) 

In [49]:
dfs = load_friends_data_raw()

[2026-06-09 11:04:47] INFO - Cargando datasets desde: H:\Cursos\Adalab_Analista & IA\taller_git\friends-analytics-workflow\data_raw
[2026-06-09 11:04:47] INFO - → Cargando weddings_divorces_ross.csv...
[2026-06-09 11:04:47] INFO - → Cargando friends_cameos.csv...
[2026-06-09 11:04:47] INFO - → Cargando friends_emotions.csv...
[2026-06-09 11:04:47] INFO - → Cargando friends_episodes.csv...
[2026-06-09 11:04:47] INFO - → Cargando friends_sets.csv...
[2026-06-09 11:04:47] INFO - → Cargando friends_info.csv...
[2026-06-09 11:04:47] INFO - → Cargando friends_quotes.csv...
[2026-06-09 11:04:47] INFO - → Cargando friends.csv...
[2026-06-09 11:04:48] INFO - → Cargando phoebe_buffay_songs.csv...
[2026-06-09 11:04:48] INFO - → Cargando duck_and_chicken.csv...
[2026-06-09 11:04:48] INFO - Todos los datasets fueron cargados correctamente.


## 1. Transformación de  las variables numéricas (friends_quotes) de orden de float a entero (int) para mejorar la estructura. 

In [50]:
df_quotes = dfs["quotes"]

df_quotes.head()

,author,episode_number,episode_title,quote,quote_order,season
0,Monica,1.0,Monica Gets A Roommate,There's nothing to tell! He's just some guy I ...,0.0,1.0
1,Joey,1.0,Monica Gets A Roommate,"C'mon, you're going out with the guy! There's ...",1.0,1.0
2,Chandler,1.0,Monica Gets A Roommate,"All right Joey, be nice. So does he have a hum...",2.0,1.0
3,Phoebe,1.0,Monica Gets A Roommate,"Wait, does he eat chalk?",3.0,1.0
4,Phoebe,1.0,Monica Gets A Roommate,"Just, 'cause, I don't want her to go through w...",4.0,1.0


In [51]:
df_quotes["quote_order"] = df_quotes["quote_order"].astype(int)
df_quotes["season"] = df_quotes["season"].astype(int)
df_quotes["episode_number"] = df_quotes["episode_number"].astype(int)


In [52]:
df_quotes.head()

,author,episode_number,episode_title,quote,quote_order,season
0,Monica,1,Monica Gets A Roommate,There's nothing to tell! He's just some guy I ...,0,1
1,Joey,1,Monica Gets A Roommate,"C'mon, you're going out with the guy! There's ...",1,1
2,Chandler,1,Monica Gets A Roommate,"All right Joey, be nice. So does he have a hum...",2,1
3,Phoebe,1,Monica Gets A Roommate,"Wait, does he eat chalk?",3,1
4,Phoebe,1,Monica Gets A Roommate,"Just, 'cause, I don't want her to go through w...",4,1


In [53]:
df_quotes.to_csv("../data_processed/friends_quotes.csv", index=False, encoding="utf-8")

## 2. Limpiar y estandarizar la columna written_by (friends_info)

In [54]:
df_info= dfs["info"]

df_info.head()

,season,episode,title,directed_by,written_by,air_date,us_views_millions,imdb_rating
0,1,1,The Pilot,James Burrows,David Crane & Marta Kauffman,1994-09-22,21.5,8.3
1,1,2,The One with the Sonogram at the End,James Burrows,David Crane & Marta Kauffman,1994-09-29,20.2,8.1
2,1,3,The One with the Thumb,James Burrows,Jeffrey Astrof & Mike Sikowitz,1994-10-06,19.5,8.2
3,1,4,The One with George Stephanopoulos,James Burrows,Alexa Junge,1994-10-13,19.7,8.1
4,1,5,The One with the East German Laundry Detergent,Pamela Fryman,Jeff Greenstein & Jeff Strauss,1994-10-20,18.6,8.5


In [55]:
trans.process_friends_writers(df_info)

¡Fichero corregido con éxito! Guardado en: H:\Cursos\Adalab_Analista & IA\taller_git\friends-analytics-workflow\data_processed\writers.csv (303 filas).


,season,episode,writer
0,1,1,David Crane
1,1,1,Marta Kauffman
2,1,2,David Crane
3,1,2,Marta Kauffman
4,1,3,Jeffrey Astrof
...,...,...,...
298,10,16,Ted Cohen
299,10,17,Marta Kauffman
300,10,17,David Crane
301,10,18,Marta Kauffman


In [56]:
df_info.drop("written_by", axis=1, inplace=True)


df_info.head(2)

,season,episode,title,directed_by,air_date,us_views_millions,imdb_rating
0,1,1,The Pilot,James Burrows,1994-09-22,21.5,8.3
1,1,2,The One with the Sonogram at the End,James Burrows,1994-09-29,20.2,8.1


In [57]:
df_info.to_csv("../data_processed/friends_info.csv", index=False, encoding="utf-8") 

#### Traducir las columnas

In [58]:
df_dac = dfs["dac"]

In [59]:
df_dac = stan.standardize_columns(df_dac,0)

In [60]:
df_dac.head()

,season,episode_number,animal,accion
0,3,3x21,Pollito,Joey lo compra
1,3,3x22,Pollito,Joey y Chandler cuidan de él.
2,3,3x22,Pato,Chandler lo rescata para que el pollito tenga ...
3,3,3x25,Pollito,Aparecen en el apartamento de los chicos.
4,3,3x25,Pato,Aparecen en el apartamento de los chicos.


In [61]:
df_dac.to_csv("../data_processed/duck_and_chicken.csv", index=False, encoding="utf-8")

In [62]:
df_cameos = dfs["cameos"]
df_cameos.head(1)

,Actor/Actriz,Personaje,Descripción/Temporada
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel (T8)


In [63]:
# 1. Extraemos la descripción y el número de la temporada usando Regex
# El patrón busca "T" seguido de uno o más números d+ dentro de un paréntesis
df_extracted = df_cameos["Descripción/Temporada"].str.extract(r"(?P<descripcion>.*?)\s*\(T(?P<temporada>\d+)\)")

# 2. Asignamos los resultados de vuelta a nuestro DataFrame original
df_cameos["descripcion"] = df_extracted["descripcion"]
df_cameos["temporada"] = df_extracted["temporada"]

# 3. Borramos la columna vieja que ya no necesitamos
df_cameos = df_cameos.drop(columns=["Descripción/Temporada"])

# Ver el resultado
df_cameos.head(1)

,Actor/Actriz,Personaje,descripcion,temporada
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel,8


In [64]:
df_cameos = stan.standardize_columns(df_cameos,0)

In [65]:
df_cameos.head()

,actor,character,description,season
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel,8
1,Bruce Willis,Paul Stevens,Padre de Elizabeth y novio de Rachel,6
2,Julia Roberts,Susie Moss,Compañera de primaria de Chandler,2
3,Charlie Sheen,Ryan,Marinero novio de Phoebe que tiene varicela,2
4,Danny DeVito,Roy,El stripper sensible en la despedida de Phoebe,10


In [66]:
df_cameos.to_csv("../data_processed/friends_cameos.csv", index=False, encoding="utf-8")

In [67]:
df_sets = dfs["sets"]

In [68]:
df_sets = stan.standardize_columns(df_sets,0)

In [69]:
df_sets.head()

,stage,type,%_scenes,description,est_num_scenes
0,Apartamento de Monica,Principal,38,El escenario con más tiempo de pantalla (cocin...,1900
1,Central Perk,Principal,24,"El punto de encuentro social, segundo en impor...",1200
2,Apartamento de Joey,Principal,18,El apartamento frente al de Monica.,900
3,Pasillo (Hallway),Transición,5,Escenas de transición entre los dos apartament...,250
4,Apartamento de Ross,Secundario,6,"Incluye su apartamento de soltero y el de ""Ugl...",300


In [70]:
df_sets.to_csv("../data_processed/friends_sets.csv", index=False, encoding="utf-8")

### Cargamos los datasets finales traducidos y comprobamos que no hay fallos en normalizaciones

In [71]:
dfs_finales = load_friends_data_translated()

[2026-06-09 11:04:48] INFO - Cargando datasets desde: H:\Cursos\Adalab_Analista & IA\taller_git\friends-analytics-workflow\data_translated
[2026-06-09 11:04:48] INFO - → Cargando friends_weddings_divorce_ross.csv...
[2026-06-09 11:04:48] INFO - → Cargando friends_cameos.csv...
[2026-06-09 11:04:48] INFO - → Cargando friends_emotions.csv...
[2026-06-09 11:04:48] INFO - → Cargando friends_episodes.csv...
[2026-06-09 11:04:48] INFO - → Cargando friends_sets.csv...
[2026-06-09 11:04:48] INFO - → Cargando friends_info.csv...
[2026-06-09 11:04:48] INFO - → Cargando friends_quotes.csv...
[2026-06-09 11:04:48] INFO - → Cargando friends.csv...
[2026-06-09 11:04:48] INFO - → Cargando friends_songs.csv...
[2026-06-09 11:04:48] INFO - → Cargando duck_and_chicken.csv...
[2026-06-09 11:04:48] INFO - → Cargando writers.csv...
[2026-06-09 11:04:48] INFO - Todos los datasets fueron cargados correctamente.


Df Quotes traducido

In [72]:
df_quotesf = dfs_finales["quotes"]

In [73]:
df_quotesf["personaje"] = df_quotesf["personaje"].str.title()

In [74]:
df_quotesf.sample(20)

,personaje,numero_episodio,titulo_episodio,cita,orden_cita,temporada
53416,Phoebe,19,El sueño de Rachel,¡Sí! ¡Siempre y cuando sea gratis! La comida a...,235,9
44030,Monica,5,La cita de Rachel,"¡No! ¡No, no! Es totalmente incompetente. Llam...",126,8
47895,Joey,21,La clase de cocina,"¡Hombre, esto es malo! Y he tenido mi parte de...",4,8
1585,Ross,7,El apagón,"No, no, no. No estoy en la zona.",80,1
16925,Ross,21,El pollito y el pato,"Oh, fue, no, bueno...",303,3
58883,Phoebe,14,Princesa Consuela,(a la mujer detrás de ella) Este lugar es muy ...,90,10
29046,Phoebe,21,La pelota,Podríamos desayunar en la cama,285,5
3384,Carol,14,Los corazones de caramelo,Ella no.,178,1
47051,Monica,17,Las hojas de té,"Bien, ¿dónde está el CD de Kat Stevens?",66,8
7929,Ross,9,El papá de Phoebe,"Hola, Phoebs, ¿cómo te fue?",210,2


In [75]:
trans.dato_a_int(df_quotesf, "temporada")

df_quotesf.head()

,personaje,numero_episodio,titulo_episodio,cita,orden_cita,temporada
0,Monica,1,Monica consigue un compañero de cuarto,¡No hay nada que contar! ¡Es sólo un tipo con ...,0,1
1,Joey,1,Monica consigue un compañero de cuarto,"¡Vamos, vas a salir con el chico! ¡Tiene que h...",1,1
2,Chandler,1,Monica consigue un compañero de cuarto,"Muy bien Joey, sé amable. Entonces ¿tiene joro...",2,1
3,Phoebe,1,Monica consigue un compañero de cuarto,"Espera, ¿come tiza?",3,1
4,Phoebe,1,Monica consigue un compañero de cuarto,Sólo porque no quiero que ella pase por lo que...,4,1


#### Utilizar una función para detectar las lineas que no son diálogo y eliminarlas, creando un archivo nuevo y limpio.

In [76]:
pr.export_data_anomalies("../data_translated/friends_quotes.csv", "friends_quotes_errores.csv", ["personaje", "cita"] )

[INFO] No data anomalies were found in the specified target columns.


""


In [77]:
pr.sanitize_dataset_by_index("../data_translated/friends_quotes.csv", "friends_quotes_errores.csv", "friends_quotes_clean.csv", chunksize=50000)

[WARNING] Anomalies tracking file not found at: friends_quotes_errores.csv. Skipping sanitization.


False

### Mapear archivo quotes con los nombres 


In [78]:
dicc_nombres = {
    "Phoe": "Phoebe",
    "Mnca": "Monica",
    "Rach": "Rachel",
    "Chan": "Chandler",
    "Estl": "Estelle",
    "Waiter" : "Camarera"
}

In [79]:
df_quotesf["personaje"] = df_quotesf["personaje"].apply(lambda x: dicc_nombres.get(x, x))

In [80]:
df_quotesf.tail()

,personaje,numero_episodio,titulo_episodio,cita,orden_cita,temporada
60191,Chandler,17,"El último, partes I y II","Oh, todo estará bien.",581,10
60192,Rachel,17,"El último, partes I y II",(llorando) ¿Tienen que ir a la nueva casa de i...,582,10
60193,Monica,17,"El último, partes I y II",Tenemos algo de tiempo.,583,10
60194,Rachel,17,"El último, partes I y II","Bien, ¿deberíamos tomar un poco de café?",584,10
60195,Chandler,17,"El último, partes I y II",Seguro. ¿Dónde?,585,10


In [81]:
df_quotesf.to_csv("../data_translated/friends_quotes.csv", index=False, encoding="utf-8")

Df friends traducido

### Normalizamos las columnas del fichero friends

In [82]:
df_friendsf = dfs_finales["friends"]

In [83]:
df_friendsf.head()

,texto,personaje,temporada,numero_episodio,escena,orden_intervencion
0,¡No hay nada que contar! ¡Es sólo un tipo con ...,Monica Geller,1,1,1,1
1,"¡Vamos, vas a salir con el chico! ¡Tiene que h...",Joey Tribbiani,1,1,1,2
2,"Muy bien Joey, sé amable. Entonces ¿tiene joro...",Chandler Bing,1,1,1,3
3,"Espera, ¿come tiza?",Phoebe Buffay,1,1,1,4
4,"(Todos se quedan mirando, desconcertados.)",Scene Directions,1,1,1,5


In [84]:
df_friendsf = stan.standardize_columns(df_friendsf,1)

In [85]:
df_friendsf.head()

,texto,personaje,temporada,numero_episodio,escena,orden_intervencion
0,¡No hay nada que contar! ¡Es sólo un tipo con ...,Monica Geller,1,1,1,1
1,"¡Vamos, vas a salir con el chico! ¡Tiene que h...",Joey Tribbiani,1,1,1,2
2,"Muy bien Joey, sé amable. Entonces ¿tiene joro...",Chandler Bing,1,1,1,3
3,"Espera, ¿come tiza?",Phoebe Buffay,1,1,1,4
4,"(Todos se quedan mirando, desconcertados.)",Scene Directions,1,1,1,5


In [86]:
df_friendsf.to_csv("../data_translated/friends.csv", index=False, encoding="utf-8")

### normalizar fecha_estreno del fichero info aaaa-mm-dd ponerlo como dd-mm-aaaa

In [87]:
df_infof = dfs_finales["info"]

In [88]:
df_infof = trans.cambiar_formato_fecha(df_infof, "fecha_estreno")

df_infof = stan.standardize_columns(df_infof,1) #normalizamos los nombres de las columnas

df_infof.head()

,temporada,numero_episodio,titulo_episodio,director,fecha_estreno,audiencia_millones,nota_imdb
0,1,1,El de Monica consigue una compañera,James Burrows,22/09/1994,21.5,8.3
1,1,2,El del sonograma al final,James Burrows,29/09/1994,20.2,8.1
2,1,3,El del pulgar,James Burrows,10/06/1994,19.5,8.2
3,1,4,El de George Stephanopoulos,James Burrows,13/10/1994,19.7,8.1
4,1,5,El del detergente de Alemania Oriental,Pamela Fryman,20/10/1994,18.6,8.5


In [89]:
 
# 3. Guardar el resultado en un nuevo archivo para no machacar el original
df_infof.to_csv("../data_translated/friends_info.csv", index=False)
 

df dac revisarlos

In [90]:
df_dac = dfs_finales["dac"]

In [91]:
df_dac.head()

,temporada,numero_episodio,animal,accion
0,3,3x21,Pollito,Joey lo compra
1,3,3x22,Pollito,Joey y Chandler cuidan de él.
2,3,3x22,Pato,Chandler lo rescata para que el pollito tenga ...
3,3,3x25,Pollito,Aparecen en el apartamento de los chicos.
4,3,3x25,Pato,Aparecen en el apartamento de los chicos.


In [92]:
#revisamos el dataset de cameos

df_cameosf = dfs_finales["cameos"]

df_cameosf.head()

,actor,personaje,descripcion,temporada
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel,8
1,Bruce Willis,Paul Stevens,Padre de Elizabeth y novio de Rachel,6
2,Julia Roberts,Susie Moss,Compañera de primaria de Chandler,2
3,Charlie Sheen,Ryan,Marinero novio de Phoebe que tiene varicela,2
4,Danny DeVito,Roy,El stripper sensible en la despedida de solter...,10


In [94]:
#revisamos el dataset de emotions

df_emotionsf = dfs_finales["emotions"]

df_emotionsf.head()

,temporada,numero_episodio,escena,intervencion,emocion
0,1,1,4,1,Enojado
1,1,1,4,3,Neutral
2,1,1,4,4,Alegre
3,1,1,4,5,Neutral
4,1,1,4,6,Neutral


In [96]:
#revisamos el dataset de episodios

df_episodesf = dfs_finales["epiv3"]

df_episodesf.head()

,año_prod,temporada,numero_episodio,titulo_episodio,duracion,resumen,director,estrellas,votos
0,1994,1,1,El de Monica consigue una compañera,22,Monica y el grupo presentan a Rachel el 'mundo...,James Burrows,8.3,7440
1,1994,1,2,El del sonograma al final,22,Ross descubre que su exesposa está embarazada....,James Burrows,8.1,4888
2,1994,1,3,El del pulgar,22,Monica se irrita cuando a todos les gusta su n...,James Burrows,8.2,4605
3,1994,1,4,El de George Stephanopoulos,22,Joey y Chandler llevan a Ross a un partido de ...,James Burrows,8.1,4468
4,1994,1,5,El del detergente de Alemania Oriental,22,"Deseoso de pasar tiempo con Rachel, Ross finge...",Pamela Fryman,8.5,4438


In [97]:
#revisamos el dataset de sets

df_setsf = dfs_finales["sets"]

df_setsf.head()

,escenario,tipo,%_escenas,descripcion,num_est_escenas
0,Apartamento de Monica,Principal,38,El escenario con más tiempo en pantalla (cocin...,1900
1,Apartamento de Joey,Principal,18,El apartamento frente al de Monica.,900
2,Pasillo,Transición,5,Escenas de transición entre los dos apartament...,250
3,Apartamento de Ross,Secundario,6,"Incluye su apartamento de soltero y el de ""Ugl...",300
4,Apartamento de Phoebe,Secundario,4,Aparece menos debido a que Phoebe vive más lejos.,200


In [98]:
#revisamos el dataset de songs

df_songsf = dfs_finales["songs"]

df_songsf.head()

,temporada,numero_episodio,cancion,descripcion
0,1,1x01,Your Love,Tu amor es como una paloma gigante...
1,1,1x10,Snowman,¿Cómo iba a saber que mi madre estaba muerta e...
2,1,1x23,Babies,Son pequeños y regordetes y tan dulces de tocar.
3,2,2x06,Smelly Cat,Su mayor éxito. ¿Qué te están dando de comer?
4,2,2x06,Terry's a Jerk,¡Terry es un imbécil y no me deja trabajar!


In [99]:
#revisamos el dataset de weddings_divorce_ross

df_weddings_divorce_rossf = dfs_finales["weddings"]

df_weddings_divorce_rossf.head()

,pareja,evento,temporada,detalle
0,Carol Willick,Boda,0,Ocurre antes del piloto (flashbacks).
1,Carol Willick,Divorcio,1,Ella descubre que es lesbiana.
2,Emily Waltham,Boda,4,Boda en Londres; Ross dice el nombre de Rachel.
3,Emily Waltham,Divorcio,5,Emily le pide no ver a Rachel y él no cumple.
4,Rachel Green,Boda,5,Se casan borrachos en Las Vegas.


In [100]:
#revisamos el dataset de writers
#  
df_writersf = dfs_finales["writers"]

df_writersf.head()

,temporada,numero_episodio,guionista
0,1,1,David Crane
1,1,1,Marta Kauffman
2,1,2,David Crane
3,1,2,Marta Kauffman
4,1,3,Jeffrey Astrof


### Planteamiento bbdd

- df_dac  							temporada	numero_episodio	animal	accion
- df_cameosf 						actor	personaje	descripcion	temporada
- df_emotionsf 						temporada	numero_episodio	escena	intervencion	emocion
- df_episodesf 						año_prod	temporada	numero_episodio	titulo_episodio	duracion	resumen	director	estrellas	votos
- df_infof 							temporada	numero_episodio	titulo_episodio	director	fecha_estreno	audiencia_millones	nota_imdb
- df_quotesf 						personaje	numero_episodio	titulo_episodio	cita	orden_cita	temporada
- (NR) df_setsf 					escenario	tipo	%_escenas	descripcion	num_est_escenas
- df_songsf							temporada	numero_episodio	cancion	descripcion
- df_weddings_divorce_rossf         pareja	evento	temporada	detalle
- df_writersf						temporada	numero_episodio	guionista
- df_friendsf						texto	personaje	temporada	numero_episodio	escena	orden_intervencion

Tenemos un esquema claro de estrella con df_episodesf e df_infof como núcleo central.

Primero, debemos fusionar las dos tablas de episodios:

df_episodesf + df_infof comparten temporada + numero_episodio + titulo_episodio + director → hacer un merge y cargar como una sola tabla dim_episodios.

Estructura recomendada
Tienes un esquema claro de estrella con df_episodesf e df_infof como núcleo central.
Primero, fusiona las dos tablas de episodios:
df_episodesf + df_infof comparten temporada + numero_episodio + titulo_episodio + director → hacer un merge y cargar como una sola tabla dim_episodios.

Tablas de dimensión
Tabla SQLFuenteClave primaria sugeridadim_episodiosepisodesf + infofid_episodio (surrogate)dim_personajescameosf (actor+personaje)id_personajedim_escenariossetsfid_escenario

Tablas de hechos
Tabla SQLFuenteFK necesariasfact_quotesquotesfid_episodio, id_personajefact_friendsfriendsfid_episodio, id_personajefact_emotionsemotionsfid_episodiofact_dacdacid_episodiofact_songssongsfid_episodiofact_writerswritersfid_episodiofact_weddingsweddings_divorce_rossf—

Recomendaciones concretas
1. Crea surrogates antes de exportar a SQL:
pythondf_episodesf["id_episodio"] = df_episodesf.groupby(
    ["temporada", "numero_episodio"]
).ngroup() + 1
Así todas las fact tables pueden hacer JOIN limpio por id_episodio.
2. df_setsf marcado como NR — revisa si %_escenas y num_est_escenas son datos reales o estimados antes de incluirlo en el modelo.
3. df_cameosf tiene actor y personaje como columnas separadas → en dim_personajes puedes tener ambos campos y distinguir personaje regular vs cameo con una columna es_cameo boolean.
4. Para el export a SQL desde Pandas:
pythonfrom sqlalchemy import create_engine

engine = create_engine("sqlite:///friends_analytics.db")  # o postgresql://...

df_episodesf.to_sql("dim_episodios", engine, if_exists="replace", index=False)
fact_quotes.to_sql("fact_quotes", engine, if_exists="replace", index=False)
# etc.
5. PowerBI/Tableau leen SQLite perfectamente para prototipo. Si el proyecto escala, migra a PostgreSQL cambiando solo el connection string.Has dicho: ¿qué son los surrogates y para qué sirve?¿qué son los surrogates y para qué sirve?11:37Claude ha respondido: Surrogate Keys (Claves Sustitutas)Surrogate Keys (Claves Sustitutas)
Una surrogate key es un identificador artificial (1, 2, 3...) que creas tú, sin significado de negocio, cuyo único propósito es identificar una fila de forma única.
Se opone a la natural key, que es un identificador que ya existe en los datos.

Ejemplo con tu caso
En tu dataset, un episodio se identifica por dos columnas:
temporadanumero_episodiotitulo11The One Where Monica Gets a Roommate12The One with the Sonogram
Para relacionar esto con fact_quotes, tendrías que hacer JOIN por dos columnas siempre:
sql-- Sin surrogate: JOIN por dos columnas en cada consulta
SELECT * FROM fact_quotes q
JOIN dim_episodios e 
  ON q.temporada = e.temporada 
  AND q.numero_episodio = e.numero_episodio
Con surrogate, añades una columna id_episodio:
id_episodiotemporadanumero_episodiotitulo111The One Where...212The One with...
sql-- Con surrogate: JOIN simple por una columna
SELECT * FROM fact_quotes q
JOIN dim_episodios e ON q.id_episodio = e.id_episodio

Por qué usarlos
Natural KeySurrogate KeyJOINmúltiples columnasuna sola columnaRendimientomás lentomás rápidoPowerBI/Tableaurelaciones complejasrelaciones simplesSi cambia el datorompe relacionesno afecta
En tu modelo, temporada + numero_episodio aparece en 7 tablas distintas — sin surrogate tendrías que repetir ese JOIN doble en cada relación del modelo.Has dicho: me parece bien lo del surrogate.me parece bien lo del surrogate. ¿como lo tendría que hacer?11:38Claude ha respondido: Para tu caso concreto, el flujo sería:Para tu caso concreto, el flujo sería:
1. Crear el surrogate en dim_episodios (la tabla maestra):
pythondf_episodios = df_episodesf.merge(df_infof, on=["temporada", "numero_episodio"], suffixes=("", "_drop"))
df_episodios = df_episodios[[c for c in df_episodios.columns if not c.endswith("_drop")]]

# Crear el surrogate
df_episodios["id_episodio"] = range(1, len(df_episodios) + 1)
2. Crear un mapa (temporada, numero_episodio) → id_episodio:
pythonmapa_episodios = df_episodios.set_index(["temporada", "numero_episodio"])["id_episodio"].to_dict()
3. Propagar el surrogate a todas las fact tables que tengan esas columnas:
pythontablas_con_episodio = [df_quotesf, df_friendsf, df_emotionsf, 
                       df_dac, df_songsf, df_writersf]

for df in tablas_con_episodio:
    df["id_episodio"] = df.set_index(["temporada", "numero_episodio"]).index.map(mapa_episodios)
4. Verificar que no hay nulos (si hay, indica que algún episodio no cruzó bien):
pythonfor df in tablas_con_episodio:
    nulos = df["id_episodio"].isna().sum()
    if nulos > 0:
        print(f"{df} tiene {nulos} filas sin id_episodio")
5. Exportar a SQL:
pythonengine = create_engine("sqlite:///friends_analytics.db")

df_episodios.to_sql("dim_episodios", engine, if_exists="replace", index=False)

df_quotesf.to_sql("fact_quotes", engine, if_exists="replace", index=False)
df_friendsf.to_sql("fact_friends", engine, if_exists="replace", index=False)
# ... resto de tablas
El orden importa: siempre cargar primero dim_episodios antes que las fact tables, ya que estas referencian su id_episodio.